In this assignment, you will be implementing two clustering validation measures: Normalized Mutual Information (NMI) and Jaccard similarity.  

You will be given one ground-truth clustering (partition) results and five clustering test cases.  You need to evaluate the clustering test cases with regard to the ground-truth by NMI and Jaccard measures and submit your measures.  You will be graded based on whether your measures are correct.  

The ground-truth clustering (partition) results are stored in file "partitions.txt"; the five clustering result test cases are stored in file "clustering_1.txt", ..., "clustering_5.txt".  

All files including partitions.txt, clustering_1.txt, ..., can be downloaded from the data.zip file attached below.

Each clustering result (both ground-truth and test cases) is represented by a file.  Each line in a file consists of two integers, separated by a space.  The first integer represents the id of a data item, and the second integer represents the id of the cluster which this item belongs to.  

You need to submit a file titled "scores.txt" consisting of 5 lines.  Each line contains two float numbers separated by a space.  The first number of the i-th line represents the NMI measure you calculated for the i-th test case i (i.e. "clustering_i.txt") with regard to the ground-truth given in "partitions.txt", and the second number of the i-th line represents the Jaccard measure you calculated for the i-th test case. 

As an example, a valid submission may look like:

```
1|  0.1000000 0.2000000
2|  0.3000000 0.4000000
3|  0.5000000 0.6000000
4|  0.7000000 0.8000000
5|  0.9000000 1.0000000
```

You will be graded based on whether your file format is correct and onhow many measures you submitted are correct.  

How to submit

When you're ready to submit, you can upload files for each part of the assignment on the "My submissions" tab.

## Solution: Compute NMI and Jaccard

The code cell below:
1. Reads the ground-truth partition from `data/partitions.txt`.
2. Reads each test clustering file `data/clustering_1.txt` to `data/clustering_5.txt`.
3. Computes **NMI** (Normalized Mutual Information) and **Jaccard similarity** (pair-count based).
4. Writes the required output file `scores.txt` with 5 lines in the requested format.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from math import log, sqrt


def read_partition(file_path):
    """Read clustering file: each line is '<item_id> <cluster_id>'"""
    mapping = {}
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            item_id, cluster_id = line.split()
            mapping[int(item_id)] = int(cluster_id)
    return mapping


def contingency_counts(true_labels, pred_labels):
    """Build cluster marginals and contingency counts from two aligned label lists."""
    a_counts = Counter(true_labels)
    b_counts = Counter(pred_labels)

    n_ij = defaultdict(int)
    for t, p in zip(true_labels, pred_labels):
        n_ij[(t, p)] += 1

    return a_counts, b_counts, n_ij


def nmi_score_from_labels(true_labels, pred_labels):
    n = len(true_labels)
    if n == 0:
        return 0.0

    a_counts, b_counts, n_ij = contingency_counts(true_labels, pred_labels)

    # Mutual Information
    mi = 0.0
    for (t, p), nij in n_ij.items():
        pij = nij / n
        pi = a_counts[t] / n
        pj = b_counts[p] / n
        mi += pij * log(pij / (pi * pj))

    # Entropies
    h_true = -sum((cnt / n) * log(cnt / n) for cnt in a_counts.values())
    h_pred = -sum((cnt / n) * log(cnt / n) for cnt in b_counts.values())

    denom = sqrt(h_true * h_pred)
    if denom == 0:
        return 0.0
    return mi / denom


def comb2(x):
    return x * (x - 1) // 2


def jaccard_score_from_labels(true_labels, pred_labels):
    """Pair-counting Jaccard: SS / (SS + SD + DS)."""
    a_counts, b_counts, n_ij = contingency_counts(true_labels, pred_labels)

    ss = sum(comb2(v) for v in n_ij.values())
    p_true = sum(comb2(v) for v in a_counts.values())
    p_pred = sum(comb2(v) for v in b_counts.values())

    denom = p_true + p_pred - ss
    if denom == 0:
        return 0.0
    return ss / denom


# Paths
base = Path("data")
ground_truth_path = base / "partitions.txt"
test_paths = [base / f"clustering_{i}.txt" for i in range(1, 6)]

# Read ground truth
ground_truth = read_partition(ground_truth_path)
item_ids = sorted(ground_truth.keys())

results = []
for test_path in test_paths:
    test_map = read_partition(test_path)

    if set(test_map.keys()) != set(item_ids):
        missing = set(item_ids) - set(test_map.keys())
        extra = set(test_map.keys()) - set(item_ids)
        raise ValueError(
            f"Mismatched item IDs in {test_path.name}. "
            f"Missing: {len(missing)}, Extra: {len(extra)}"
        )

    y_true = [ground_truth[i] for i in item_ids]
    y_pred = [test_map[i] for i in item_ids]

    nmi = nmi_score_from_labels(y_true, y_pred)
    jac = jaccard_score_from_labels(y_true, y_pred)
    results.append((nmi, jac))

# Write required submission file
out_path = Path("scores.txt")
with open(out_path, "w", encoding="utf-8") as f:
    for nmi, jac in results:
        f.write(f"{nmi:.7f} {jac:.7f}\n")

print(f"Saved: {out_path.resolve()}")
print("\nScores (NMI, Jaccard):")
for i, (nmi, jac) in enumerate(results, start=1):
    print(f"{i}: {nmi:.7f} {jac:.7f}")